# Predicting Emerging Community Bridges in Reddit Using Social Network Analysis and Link Prediction

## Introduction
Social networks are highly dynamic environments where communities constantly evolve, interact, and form new structural bridges. On platforms like Reddit, individual communities (subreddits) frequently reference one another via hyperlinks, creating a massive, complex network of information flow, cultural exchange, and sometimes conflict.

## Problem Definition
Understanding how and why digital communities interact is crucial for modeling information diffusion, measuring platform polarization, and predicting the formation of echo chambers. The core problem this project addresses is **Community Bridge Prediction**: Given a static snapshot of subreddit-to-subreddit interactions, can we predict which communities are most likely to form *new* hyperlinks with each other in the future, relying purely on the underlying topological structure of the network?

## Objectives
1. **Network Construction:** Project a massive, temporal, directed hyperlink dataset into a clean, undirected, and unweighted graph suitable for foundational topological analysis.
2. **Exploratory SNA:** Analyze the macro and micro-level structural properties of the Reddit network (e.g., degree distribution, density, connected components).
3. **Link Prediction:** Formulate bridge prediction as a Link Prediction problem, implementing and comparing traditional topological heuristics: *Common Neighbors*, *Jaccard Coefficient*, *Adamic-Adar Index*, and *Preferential Attachment*.
4. **Evaluation:** Rigorously evaluate these heuristics using a held-out test set and negative sampling to identify the most predictive topological features of community bridging.

## SDG Mapping
This project aligns with the United Nations Sustainable Development Goals (SDGs):
* **SDG 16 (Peace, Justice and Strong Institutions):** By mapping and predicting how different communities interact, we can better understand digital polarization, track the spread of misinformation across community bridges, and identify isolated echo chambers, contributing to healthier digital institutions.
* **SDG 9 (Industry, Innovation and Infrastructure):** Developing better models for information routing and social network topology improves the algorithmic infrastructure of modern digital communication platforms.

## Dataset Description
This project utilizes the official **Stanford SNAP soc-RedditHyperlinks-title dataset**. 
* **Nodes:** 55,863 subreddits.
* **Edges:** 858,490 hyperlinks originating from a post in a source subreddit to a post in a target subreddit.
* **Properties:** The raw dataset is directed, temporal, signed (sentiment), and highly attributed (textual features). 
* **Transformation:** For this specific core experiment focusing on pure topological link prediction, the dataset is systematically projected into a static, undirected, unweighted network by collapsing multi-edges and removing self-loops. This ensures that baseline algorithms like Adamic-Adar and Jaccard Coefficient can be evaluated consistently.

---
## Data Preprocessing

In [ ]:
# Run this cell ONCE to install the required packages into your active base Conda environment
!pip install -r ../requirements.txt

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import data_preprocessing as dp
import network_analysis as na
import link_prediction as lp
import pprint

# Dataset Path
dataset_path = '../data/raw/soc-redditHyperlinks-title.tsv'

### Load and Validate Data

In [ ]:
df = dp.load_dataset(dataset_path)
print(df.head())

validation_results = dp.basic_validation(df)
pprint.pprint(validation_results)

## Network Construction
Constructing an undirected unweighted graph for the core experiment. Self-loops are removed and duplicate pairs are collapsed.

In [ ]:
G = na.construct_undirected_graph(df)

## Exploratory SNA
Calculating basic network statistics.

In [ ]:
stats = na.get_network_statistics(G)
pprint.pprint(stats)

## Community Detection

## Train/Test Split
Using approximately 80% positive edges for training and 20% held-out edges for testing. The test edges are removed from the training graph.

In [ ]:
G_train, train_edges_pos, test_edges_pos = lp.train_test_split_edges(G, test_fraction=0.2, seed=42)

## Negative Sampling
Creating valid non-edge negative examples. Avoiding self-loops and existing positive edges.

In [ ]:
# To balance the classes, we sample as many negative edges as there are positive edges in the test set
num_neg_samples = len(test_edges_pos)
test_edges_neg = lp.generate_negative_samples(G, num_neg_samples, seed=42)

# Save processed graph and edges for downstream modules to prevent recomputing
dp.save_processed_data(G_train, train_edges_pos, test_edges_pos, test_edges_neg)

## Link Prediction
Implementing basic topological link prediction functions: Common Neighbors, Jaccard Coefficient, Adamic-Adar, and Preferential Attachment. We calculate scores on the held-out test edges and negative samples.

In [ ]:
candidate_edges = test_edges_pos + test_edges_neg

cn_scores = lp.predict_common_neighbors(G_train, candidate_edges)
jc_scores = lp.predict_jaccard(G_train, candidate_edges)
aa_scores = lp.predict_adamic_adar(G_train, candidate_edges)
pa_scores = lp.predict_preferential_attachment(G_train, candidate_edges)

print("Link prediction scoring complete.")

## Evaluation
## Top-10/20
## Community Bridge Analysis
## Cytoscape
## Interpretation
## Limitations
## Conclusion